# **準備：Google Drive をマウント**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **準備：環境構築**

In [4]:
# 出力を無効化
import os, sys

# 必要なライブラリをインストール
!pip install transformers datasets scikit-learn
!pip install fugashi unidic-lite

# GPU確認
import torch; torch.cuda.is_available(), torch.cuda.get_device_name(0)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

(True, 'Tesla T4')

# **検証：学習データの準備** (ここまで毎回実行)

In [ ]:
# Google Drive に保存しているので、 アップロード不要
from google.colab import files; uploaded = files.upload()

In [6]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset

# 1. CSV読み込み
selected_file = "/content/drive/MyDrive/Colab_Notebooks/csv/role_classification.csv"
df = pd.read_csv(selected_file)

# 2. ラベルを数値に変換
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["役割ラベル"])

# 3. Datasetへ変換
dataset = Dataset.from_pandas(df[["発言内容", "label"]])

# 4. 結果確認（任意）
print(f"\n✅ {selected_file} を読み込みました。")

df["役割ラベル"].value_counts()


✅ /content/drive/MyDrive/Colab_Notebooks/csv/role_classification.csv を読み込みました。


,count
役割ラベル,
ファシリテーター,143
タイムキーパー,143
アイデアマン,143
クリティカルシンカー,143
リスナー,143
書記,143
調整役,143


# **検証：トークナイザー・トークナイズ**

In [4]:
from transformers import BertJapaneseTokenizer, BertForSequenceClassification

# トークナイザーとモデルのロード
tokenizer = BertJapaneseTokenizer.from_pretrained("cl-tohoku/bert-base-japanese-v3")
model = BertForSequenceClassification.from_pretrained(
    "cl-tohoku/bert-base-japanese-v3",
    num_labels=len(label_encoder.classes_)
)

# トークナイズ関数
def tokenize_function(examples):
    return tokenizer(examples["発言内容"], padding="max_length", truncation=True, max_length=512)

# トークナイズ
tokenized_dataset = dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/251 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/231k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/447M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cl-tohoku/bert-base-japanese-v3 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Parameter 'function'=<function tokenize_function at 0x7fc56eb6fe20> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

# **検証：学習処理（Trainer）**

In [6]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab_Notebooks/results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    save_strategy="epoch",
    logging_dir="/content/drive/MyDrive/Colab_Notebooks/logs",
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,
)

# 学習スタート
trainer.train()

# 必ず次の「モデル保存＋評価」を行ってください

<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yujisk (yujisk-international-professional-university-of-technolo) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,1.812400
20,1.282000
30,0.750700
40,0.325300
50,0.304700
60,0.064400
70,0.032600
80,0.088500
90,0.148800
100,0.054600


TrainOutput(global_step=630, training_loss=0.09292147106431897, metrics={'train_runtime': 662.9567, 'train_samples_per_second': 7.55, 'train_steps_per_second': 0.95, 'total_flos': 1316929950336000.0, 'train_loss': 0.09292147106431897, 'epoch': 5.0})

# **検証：Google Drive に保存＋評価**


In [7]:
import pickle

model.save_pretrained("/content/drive/MyDrive/Colab_Notebooks/role_classifier_model", safe_serialization=False)
tokenizer.save_pretrained("/content/drive/MyDrive/Colab_Notebooks/role_classifier_model")
with open("/content/drive/MyDrive/Colab_Notebooks/role_classifier_model/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

# モデル評価
trainer.evaluate()


{'eval_loss': 0.0005246535874903202,
 'eval_runtime': 28.5979,
 'eval_samples_per_second': 35.003,
 'eval_steps_per_second': 4.406,
 'epoch': 5.0}

# **推論：単文で分類**

In [8]:
import torch

# モデルとトークナイザーのロード
from transformers import AutoTokenizer, AutoModelForSequenceClassification
model_path = "/content/drive/MyDrive/Colab_Notebooks/role_classifier_model"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

text = "そろそろ次の議題に移りましょうか？"
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

outputs = model(**inputs)
pred = outputs.logits.argmax(dim=1)
probs = torch.nn.functional.softmax(outputs.logits, dim=1)
confidence = probs[0][pred].item()
label = label_encoder.inverse_transform(pred.cpu().numpy())

print(f"予測ラベル: {label[0]}, 信頼度: {confidence:.2f}")


予測ラベル: タイムキーパー, 信頼度: 1.00


# **推論：複数発言の分類**

In [8]:
# モデルとトークナイザーのロード
from transformers import AutoTokenizer, AutoModelForSequenceClassification
model_path = "/content/drive/MyDrive/Colab_Notebooks/role_classifier_model"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

test_sentences = [
    "この日程で皆さん大丈夫でしょうか？調整も可能です。",
    "その件でいうと、近年関連のニュースがよくでますよね",
    "ご意見ある方いらっしゃいますか？遠慮なくどうぞ。",
    "その点については、別の視点からも検討が必要ですね。",
    "資料の5ページをご覧ください。こちらに詳しく書いてあります。",
    "前回の会議で出た課題は解決しましたか？",
    "スケジュール的に厳しいかもしれませんが、なんとか調整してみます。",
    "この案に反対の方はいらっしゃいますか？",
    "ちょっと確認させていただきたいのですが、意思決定はどのタイミングでしょうか？",
    "その話は一旦ここで区切って、次の議題に移りましょう。",
    "今のお話をまとめると、こういう方向性で進めていくということでよろしいでしょうか？"
]

for text in test_sentences:
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    pred = probs.argmax(dim=1)
    confidence = probs[0][pred].item()
    label = label_encoder.inverse_transform(pred.cpu().numpy())
    print(f"発言: {text}\n→ 予測ラベル: {label[0]}, 信頼度: {confidence:.2f}\n")


発言: この日程で皆さん大丈夫でしょうか？調整も可能です。
→ 予測ラベル: 調整役, 信頼度: 1.00

発言: その件でいうと、近年関連のニュースがよくでますよね
→ 予測ラベル: ファシリテーター, 信頼度: 0.89

発言: ご意見ある方いらっしゃいますか？遠慮なくどうぞ。
→ 予測ラベル: ファシリテーター, 信頼度: 1.00

発言: その点については、別の視点からも検討が必要ですね。
→ 予測ラベル: クリティカルシンカー, 信頼度: 1.00

発言: 資料の5ページをご覧ください。こちらに詳しく書いてあります。
→ 予測ラベル: 書記, 信頼度: 1.00

発言: 前回の会議で出た課題は解決しましたか？
→ 予測ラベル: ファシリテーター, 信頼度: 0.72

発言: スケジュール的に厳しいかもしれませんが、なんとか調整してみます。
→ 予測ラベル: 調整役, 信頼度: 1.00

発言: この案に反対の方はいらっしゃいますか？
→ 予測ラベル: ファシリテーター, 信頼度: 0.99

発言: ちょっと確認させていただきたいのですが、意思決定はどのタイミングでしょうか？
→ 予測ラベル: ファシリテーター, 信頼度: 0.99

発言: その話は一旦ここで区切って、次の議題に移りましょう。
→ 予測ラベル: ファシリテーター, 信頼度: 1.00

発言: 今のお話をまとめると、こういう方向性で進めていくということでよろしいでしょうか？
→ 予測ラベル: ファシリテーター, 信頼度: 1.00

